In [2]:
"""
Self-contained fidelity–shot-cost tradeoff analysis and figure generation.

Inputs  : dynamic_circuit_results.json   (from dynamic_circuit_pipeline.py)
Outputs : fig_tradeoff1_F_vs_Nuseful.pdf
          fig_tradeoff2_F_vs_Ntotal.pdf
          fig_tradeoff3_sigma_vs_N.pdf
          fig_tradeoff_panel.pdf

No external data files needed — subsampling runs here.
"""

import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

# =============================================================================
#  CONFIG — set paths here
# =============================================================================
DYNAMIC_JSON = "/Users/nandan/Desktop/CTCs/IBM/dynamic_circuit_results.json"   # <-- update path if needed
OUT_DIR      = "./"                              # where to save PDFs
N_SWEEP      = [200, 400, 600, 800, 1000, 1500,
                2000, 3000, 4000, 5000, 6000,
                7000, 8000, 9000, 10000]
N_REPEATS    = 30
SEED         = 42

# =============================================================================
#  STEP 1 — Load hardware data
# =============================================================================
with open(DYNAMIC_JSON) as f:
    D = json.load(f)

THETA  = D['metadata']['theta_msg']
VARPHI = D['metadata']['varphi_msg']
shots  = D['metadata']['shots']
rc     = D['postselected']['raw_counts']   # ps_Z/X/Y, dyn_Z/X/Y
p_succ = D['postselected']['p_succ']
trd    = D['tradeoff']

ps_sum  = {'F'    : D['postselected']['fidelity'],
            'F_ci' : D['postselected']['fidelity_ci'],
            'n_kept': D['postselected']['n_kept']}
dyn_sum = {'F'    : D['dynamic']['fidelity'],
            'F_ci' : D['dynamic']['fidelity_ci'],
            'n_kept': D['dynamic']['n_kept']}

print("=" * 60)
print("  Fidelity–Shot-Cost Tradeoff Analysis")
print("=" * 60)
print(f"  Backend : {D['metadata']['backend']}")
print(f"  Shots   : {shots:,} per basis")
print(f"  p_succ  : {p_succ:.4f}  →  {trd['shot_cost_ratio']:.2f}× shot overhead\n")

# =============================================================================
#  STEP 2 — Build ideal ρ_M
# =============================================================================
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity

qcm = QuantumCircuit(1)
qcm.u(THETA, VARPHI, 0.0, 0)
rho_M = DensityMatrix(Statevector.from_instruction(qcm))

# =============================================================================
#  STEP 3 — Helper functions
# =============================================================================
def parse_bs(bs):
    p = bs.split(' ') if ' ' in bs else list(bs)
    return p[0], p[1], p[2]   # crTomo, crG, crR

def pauli_exp(c):
    n0 = c.get('0', 0); n1 = c.get('1', 0); N = n0 + n1
    return (n0 - n1) / N if N > 0 else 0.0

def reconstruct(sx, sy, sz):
    X = np.array([[0,1],[1,0]], dtype=complex)
    Y = np.array([[0,-1j],[1j,0]], dtype=complex)
    Z = np.array([[1,0],[0,-1]], dtype=complex)
    rho = (np.eye(2) + sx*X + sy*Y + sz*Z) / 2
    ev, evec = np.linalg.eigh(rho)
    ev = np.maximum(ev, 0); ev /= ev.sum()
    return (evec * ev) @ evec.conj().T

def compute_F_from_tomo(tomo_X, tomo_Y, tomo_Z):
    sx = pauli_exp(tomo_X)
    sy = pauli_exp(tomo_Y)
    sz = pauli_exp(tomo_Z)
    rho = reconstruct(sx, sy, sz)
    return float(state_fidelity(DensityMatrix(rho), rho_M))

def counts_to_shots(counts_dict):
    """Expand {bitstring: count} → list of bitstrings."""
    shots_list = []
    for bs, cnt in counts_dict.items():
        shots_list.extend([bs] * cnt)
    return shots_list

# =============================================================================
#  STEP 4 — Expand raw counts into per-shot lists
# =============================================================================
ps_shots  = {b: counts_to_shots(rc[f'ps_{b}'])  for b in ['Z', 'X', 'Y']}
dyn_shots = {b: counts_to_shots(rc[f'dyn_{b}']) for b in ['Z', 'X', 'Y']}

print(f"  Shots available — ps: {len(ps_shots['Z']):,}  dyn: {len(dyn_shots['Z']):,}")

# =============================================================================
#  STEP 5 — Subsampling sweep
# =============================================================================
print(f"\n  Running subsampling sweep ({N_REPEATS} repeats per point) ...")
print(f"  {'N_total':>8}  {'ps_Nkept':>10}  {'ps_F':>8}  {'ps_σ':>8}"
      f"  {'dyn_Nkept':>10}  {'dyn_F':>8}  {'dyn_σ':>8}")
print("  " + "-" * 72)

RNG = np.random.default_rng(SEED)

ps_results  = []   # list of dicts
dyn_results = []

for N_total in N_SWEEP:
    ps_F_reps = []; ps_nk_reps = []
    dyn_F_reps = []

    for _ in range(N_REPEATS):
        # ── Post-selected subsample ──────────────────────────────────────────
        idx = RNG.choice(len(ps_shots['Z']), size=N_total, replace=False)
        tomo_ps = {}
        nk = 0
        for b in ['Z', 'X', 'Y']:
            sub  = [ps_shots[b][i] for i in idx]
            kept = defaultdict(int)
            for bs in sub:
                crT, crG, crR = parse_bs(bs)
                if crR == '0' and crG == '0':
                    kept[crT] += 1
                    if b == 'Z': nk += 1
            tomo_ps[b] = dict(kept)
        ps_nk_reps.append(nk)
        if sum(tomo_ps['Z'].values()) > 5:
            ps_F_reps.append(compute_F_from_tomo(
                tomo_ps['X'], tomo_ps['Y'], tomo_ps['Z']))

        # ── Dynamic subsample ────────────────────────────────────────────────
        idx2 = RNG.choice(len(dyn_shots['Z']), size=N_total, replace=False)
        tomo_dyn = {}
        for b in ['Z', 'X', 'Y']:
            sub  = [dyn_shots[b][i] for i in idx2]
            kept = defaultdict(int)
            for bs in sub:
                crT, crG, crR = parse_bs(bs)
                kept[crT] += 1
            tomo_dyn[b] = dict(kept)
        dyn_F_reps.append(compute_F_from_tomo(
            tomo_dyn['X'], tomo_dyn['Y'], tomo_dyn['Z']))

    ps_nk_mean = float(np.mean(ps_nk_reps))
    ps_F_mean  = float(np.mean(ps_F_reps))  if ps_F_reps  else float('nan')
    ps_F_std   = float(np.std(ps_F_reps))   if ps_F_reps  else float('nan')
    dyn_F_mean = float(np.mean(dyn_F_reps))
    dyn_F_std  = float(np.std(dyn_F_reps))

    ps_results.append({'n_total': N_total, 'n_kept': ps_nk_mean,
                       'F_mean': ps_F_mean, 'F_std': ps_F_std})
    dyn_results.append({'n_total': N_total, 'n_kept': float(N_total),
                        'F_mean': dyn_F_mean, 'F_std': dyn_F_std})

    print(f"  {N_total:>8,}  {ps_nk_mean:>10.0f}  {ps_F_mean:>8.4f}  {ps_F_std:>8.4f}"
          f"  {N_total:>10,}  {dyn_F_mean:>8.4f}  {dyn_F_std:>8.4f}")

# Convert to arrays
ps_Ntot  = np.array([r['n_total'] for r in ps_results])
ps_Nkept = np.array([r['n_kept']  for r in ps_results])
ps_Fmean = np.array([r['F_mean']  for r in ps_results])
ps_Fstd  = np.array([r['F_std']   for r in ps_results])

dyn_Ntot  = np.array([r['n_total'] for r in dyn_results])
dyn_Nkept = np.array([r['n_kept']  for r in dyn_results])
dyn_Fmean = np.array([r['F_mean']  for r in dyn_results])
dyn_Fstd  = np.array([r['F_std']   for r in dyn_results])

# =============================================================================
#  STEP 6 — Plotting style
# =============================================================================
plt.rcParams.update({
    'font.family'      : 'DejaVu Serif',
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 10,
    'ytick.labelsize'  : 10,
    'legend.fontsize'  : 9.5,
    'figure.dpi'       : 180,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.linewidth'   : 0.8,
    'pdf.fonttype'     : 42,
})

C_PS    = '#C0392B'   # deep red   — post-selected
C_DYN   = '#2980B9'   # blue       — dynamic
C_IDEAL = '#27AE60'   # green      — ideal

delta_F = ps_sum['F'] - dyn_sum['F']

# =============================================================================
#  FIG 1 — F vs N_useful (kept shots)
# =============================================================================
def fig_F_vs_Nkept():
    fig, ax = plt.subplots(figsize=(6.2, 4.4))

    ax.fill_between(ps_Nkept, ps_Fmean-ps_Fstd, ps_Fmean+ps_Fstd,
                    alpha=0.18, color=C_PS)
    ax.fill_between(dyn_Nkept, dyn_Fmean-dyn_Fstd, dyn_Fmean+dyn_Fstd,
                    alpha=0.18, color=C_DYN)
    ax.plot(ps_Nkept,  ps_Fmean,  color=C_PS,  lw=2.2,
            label=f'Post-selected  ($F_{{\\infty}}={ps_sum["F"]:.4f}$)')
    ax.plot(dyn_Nkept, dyn_Fmean, color=C_DYN, lw=2.2,
            label=f'Dynamic  ($F_{{\\infty}}={dyn_sum["F"]:.4f}$)')
    ax.axhline(ps_sum['F'],  color=C_PS,  ls='--', lw=1.0, alpha=0.55)
    ax.axhline(dyn_sum['F'], color=C_DYN, ls='--', lw=1.0, alpha=0.55)
    ax.axhline(1.0, color=C_IDEAL, ls=':', lw=1.3, alpha=0.7, label='Ideal $F=1$')

    # Annotate ΔF gap
    x_ann = 2600
    ax.annotate('', xy=(x_ann, dyn_sum['F']),
                xytext=(x_ann, ps_sum['F']),
                arrowprops=dict(arrowstyle='<->', color='#555', lw=1.3))
    ax.text(x_ann + 180, (ps_sum['F']+dyn_sum['F'])/2,
            f'$\\Delta F={delta_F:.4f}$\n(feedforward\ncost)',
            fontsize=8.5, color='#333', va='center',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#bbb', lw=0.7))

    ax.set_xlabel('Useful (kept) output samples $N_{\\rm useful}$')
    ax.set_ylabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_title('Fidelity vs. useful sample count\n'
                 f'ibm\\_torino  ·  subsampled from {shots:,}-shot run  ·  '
                 f'{N_REPEATS} repeats/point', pad=8)
    ax.legend(framealpha=0.9, edgecolor='#ccc', loc='lower right')
    ax.set_xlim(0, 10500); ax.set_ylim(0.55, 1.05)
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    ax.text(0.02, 0.04, f'Bands = $\\pm 1\\sigma$ over {N_REPEATS} subsampling repeats',
            transform=ax.transAxes, fontsize=8, color='#666', style='italic')
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 2 — F vs N_total (total shots fired)
# =============================================================================
def fig_F_vs_Ntotal():
    fig, ax = plt.subplots(figsize=(6.2, 4.4))

    ax.fill_between(ps_Ntot, ps_Fmean-ps_Fstd, ps_Fmean+ps_Fstd,
                    alpha=0.18, color=C_PS)
    ax.fill_between(dyn_Ntot, dyn_Fmean-dyn_Fstd, dyn_Fmean+dyn_Fstd,
                    alpha=0.18, color=C_DYN)
    ax.plot(ps_Ntot,  ps_Fmean,  color=C_PS,  lw=2.2,
            label='Post-selected')
    ax.plot(dyn_Ntot, dyn_Fmean, color=C_DYN, lw=2.2,
            label='Dynamic (feedforward)')
    ax.axhline(1.0, color=C_IDEAL, ls=':', lw=1.3, alpha=0.7, label='Ideal $F=1$')

    # Shade ps fidelity advantage
    ax.fill_between(ps_Ntot, dyn_Fmean, ps_Fmean,
                    where=(ps_Fmean > dyn_Fmean),
                    alpha=0.09, color=C_PS,
                    label='ps fidelity advantage')

    # Annotate equal-σ crossover
    crossover_idx = np.argmin(np.abs(ps_Fstd - dyn_Fstd))
    xc = int(ps_Ntot[crossover_idx])
    ax.axvline(xc, color='#888', ls=':', lw=1.1, alpha=0.7)
    ax.text(xc + 200, 0.60,
            f'Equal $\\sigma_F$\n$N_{{\\rm total}}\\approx{xc:,}$',
            fontsize=8.5, color='#555',
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#bbb', lw=0.6))

    ax.set_xlabel('Total shots fired $N_{\\rm total}$ (per basis)')
    ax.set_ylabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_title(f'Fidelity vs. total shots fired\n'
                 f'(post-selected wastes $1-p_{{\\rm succ}}\\approx'
                 f'{100*(1-p_succ):.0f}\\%$ of shots)', pad=8)
    ax.legend(framealpha=0.9, edgecolor='#ccc', loc='lower right')
    ax.set_xlim(0, 10500); ax.set_ylim(0.55, 1.05)
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    ax.text(0.02, 0.04, f'Bands = $\\pm 1\\sigma$ over {N_REPEATS} subsampling repeats',
            transform=ax.transAxes, fontsize=8, color='#666', style='italic')
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 3 — σ_F vs N_useful
# =============================================================================
def fig_sigma_vs_Nkept():
    fig, ax = plt.subplots(figsize=(6.2, 4.4))

    ax.plot(ps_Nkept,  ps_Fstd,  color=C_PS,  lw=2.2, marker='o', ms=4.5,
            label='Post-selected')
    ax.plot(dyn_Nkept, dyn_Fstd, color=C_DYN, lw=2.2, marker='s', ms=4.5,
            label='Dynamic (feedforward)')

    # 1/sqrt(N) reference lines anchored at largest N
    N_arr = np.linspace(50, 10500, 500)
    for col, Fstd_arr, Nkept_arr in [(C_PS, ps_Fstd, ps_Nkept),
                                      (C_DYN, dyn_Fstd, dyn_Nkept)]:
        sc = Fstd_arr[-4] * np.sqrt(Nkept_arr[-4])
        ax.plot(N_arr, sc / np.sqrt(N_arr), color=col,
                lw=1.0, ls='--', alpha=0.55)

    # Annotate: equal σ_F requires 4× more shots for ps
    N_target   = 600
    sigma_dyn_t = dyn_Fstd[np.argmin(np.abs(dyn_Nkept - N_target))]
    N_ps_equiv  = int(ps_Nkept[np.argmin(np.abs(ps_Fstd - sigma_dyn_t))])
    ax.annotate('', xy=(N_target, sigma_dyn_t),
                xytext=(N_ps_equiv, sigma_dyn_t),
                arrowprops=dict(arrowstyle='<->', color='#888', lw=1.3))
    ax.text((N_target + N_ps_equiv)/2, sigma_dyn_t + 0.003,
            f'{trd["shot_cost_ratio"]:.1f}$\\times$ more useful\nshots for post-sel.',
            ha='center', fontsize=8.5, color='#444',
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#bbb', lw=0.6))

    ax.set_xlabel('Useful (kept) samples $N_{\\rm useful}$')
    ax.set_ylabel('$\\sigma_F$  (std dev of fidelity estimate)')
    ax.set_title(f'Statistical precision vs. useful sample count\n'
                 f'Dynamic achieves same $\\sigma_F$ with '
                 f'{trd["shot_cost_ratio"]:.2f}$\\times$ fewer total shots',
                 pad=8)
    ax.legend(framealpha=0.9, edgecolor='#ccc', fontsize=9.5,
              loc='upper right')
    ax.text(0.97, 0.92, '$1/\\sqrt{N}$ scaling shown dashed',
            transform=ax.transAxes, ha='right', fontsize=8.5,
            color='#666', style='italic')
    ax.set_xlim(0, 10500); ax.set_ylim(0, 0.08)
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 4 — Combined 1×3 panel
# =============================================================================
def fig_panel():
    fig = plt.figure(figsize=(15.5, 5.0))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.36)

    # ── (a) F vs N_useful ───────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    ax1.fill_between(ps_Nkept, ps_Fmean-ps_Fstd, ps_Fmean+ps_Fstd,
                     alpha=0.18, color=C_PS)
    ax1.fill_between(dyn_Nkept, dyn_Fmean-dyn_Fstd, dyn_Fmean+dyn_Fstd,
                     alpha=0.18, color=C_DYN)
    ax1.plot(ps_Nkept,  ps_Fmean,  color=C_PS,  lw=2.0,
             label=f'Post-selected ($F={ps_sum["F"]:.4f}$)')
    ax1.plot(dyn_Nkept, dyn_Fmean, color=C_DYN, lw=2.0,
             label=f'Dynamic ($F={dyn_sum["F"]:.4f}$)')
    ax1.axhline(ps_sum['F'],  color=C_PS,  ls='--', lw=0.9, alpha=0.5)
    ax1.axhline(dyn_sum['F'], color=C_DYN, ls='--', lw=0.9, alpha=0.5)
    ax1.axhline(1.0, color=C_IDEAL, ls=':', lw=1.2, alpha=0.7, label='Ideal')
    ax1.text(0.97, 0.18,
             f'$\\Delta F = {delta_F:.4f}$\n(feedforward cost)',
             transform=ax1.transAxes, ha='right', fontsize=9, color='#333',
             bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#bbb', lw=0.7))
    ax1.set_xlabel('Useful samples $N_{\\rm useful}$')
    ax1.set_ylabel('$F(\\rho_Y,\\, \\rho_M)$')
    ax1.set_title('(a) Fidelity vs. useful samples', fontsize=11)
    ax1.legend(framealpha=0.9, edgecolor='#ccc', fontsize=9, loc='lower right')
    ax1.set_xlim(0, 10500); ax1.set_ylim(0.55, 1.05)
    ax1.yaxis.grid(True, alpha=0.3, zorder=0)

    # ── (b) F vs N_total ────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])
    ax2.fill_between(ps_Ntot, ps_Fmean-ps_Fstd, ps_Fmean+ps_Fstd,
                     alpha=0.18, color=C_PS)
    ax2.fill_between(dyn_Ntot, dyn_Fmean-dyn_Fstd, dyn_Fmean+dyn_Fstd,
                     alpha=0.18, color=C_DYN)
    ax2.plot(ps_Ntot,  ps_Fmean,  color=C_PS,  lw=2.0, label='Post-selected')
    ax2.plot(dyn_Ntot, dyn_Fmean, color=C_DYN, lw=2.0, label='Dynamic')
    ax2.axhline(1.0, color=C_IDEAL, ls=':', lw=1.2, alpha=0.7, label='Ideal')
    ax2.fill_between(ps_Ntot, dyn_Fmean, ps_Fmean,
                     where=(ps_Fmean > dyn_Fmean),
                     alpha=0.09, color=C_PS, label='ps advantage')
    ax2.set_xlabel('Total shots $N_{\\rm total}$ (per basis)')
    ax2.set_ylabel('$F(\\rho_Y,\\, \\rho_M)$')
    ax2.set_title('(b) Fidelity vs. total shots fired', fontsize=11)
    ax2.legend(framealpha=0.9, edgecolor='#ccc', fontsize=9, loc='lower right')
    ax2.set_xlim(0, 10500); ax2.set_ylim(0.55, 1.05)
    ax2.yaxis.grid(True, alpha=0.3, zorder=0)

    # ── (c) σ_F vs N_useful ─────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[2])
    ax3.plot(ps_Nkept,  ps_Fstd,  color=C_PS,  lw=2.0,
             marker='o', ms=3.5, label='Post-selected')
    ax3.plot(dyn_Nkept, dyn_Fstd, color=C_DYN, lw=2.0,
             marker='s', ms=3.5, label='Dynamic')
    N_arr = np.linspace(50, 10500, 500)
    for col, Fs, Nk in [(C_PS, ps_Fstd, ps_Nkept),
                         (C_DYN, dyn_Fstd, dyn_Nkept)]:
        sc = Fs[-4] * np.sqrt(Nk[-4])
        ax3.plot(N_arr, sc/np.sqrt(N_arr), color=col, lw=0.9, ls='--', alpha=0.5)
    ax3.set_xlabel('Useful samples $N_{\\rm useful}$')
    ax3.set_ylabel('$\\sigma_F$')
    ax3.set_title('(c) Statistical precision\n'
                  '(dashed = $1/\\sqrt{N}$ reference)', fontsize=11)
    ax3.legend(framealpha=0.9, edgecolor='#ccc', fontsize=9)
    ax3.set_xlim(0, 10500); ax3.set_ylim(0, 0.08)
    ax3.yaxis.grid(True, alpha=0.3, zorder=0)

    fig.suptitle(
        f'Fidelity-shot-cost tradeoff: post-selected vs. dynamic  '
        f'(ibm\\_torino,  $N_{{\\rm shots}}={shots:,}$/basis)\n'
        f'$p_{{\\rm succ}}={p_succ:.4f}$  '
        f'$\\Rightarrow$ dynamic saves {trd["shot_cost_ratio"]:.2f}$\\times$ shots  '
        f'at fidelity cost $\\Delta F={delta_F:.4f}$  '
        f'({N_REPEATS} subsamples/point)',
        fontsize=11, y=1.02
    )
    fig.tight_layout()
    return fig

# =============================================================================
#  STEP 7 — Save figures
# =============================================================================
print(f"\nGenerating figures ...")
os.makedirs(OUT_DIR, exist_ok=True)
for fname, fn in [
    ('fig_tradeoff1_F_vs_Nuseful.pdf',  fig_F_vs_Nkept),
    ('fig_tradeoff2_F_vs_Ntotal.pdf',   fig_F_vs_Ntotal),
    ('fig_tradeoff3_sigma_vs_N.pdf',    fig_sigma_vs_Nkept),
    ('fig_tradeoff_panel.pdf',          fig_panel),
]:
    f = fn()
    path = os.path.join(OUT_DIR, fname)
    f.savefig(path, bbox_inches='tight', dpi=200)
    plt.close(f)
    print(f"  [✓] {fname}")

# =============================================================================
#  STEP 8 — Print paper-ready numbers
# =============================================================================
print(f"\n{'='*60}")
print("  PAPER-READY TRADEOFF NUMBERS")
print(f"{'='*60}")
print(f"  Backend          : {D['metadata']['backend']}")
print(f"  Shots per basis  : {shots:,}")
print(f"  p_succ           : {p_succ:.4f}  ({trd['shot_cost_ratio']:.2f}x shot overhead)")
print(f"")
print(f"  Post-selected:")
print(f"    F(rho_Y, rho_M) = {ps_sum['F']:.4f}  "
      f"[{ps_sum['F_ci'][0]:.4f}, {ps_sum['F_ci'][1]:.4f}]")
idx2500 = np.argmin(np.abs(ps_Nkept - 2500))
print(f"    sigma_F @ N=2500 = {ps_Fstd[idx2500]:.4f}")
print(f"")
print(f"  Dynamic (feedforward):")
print(f"    F(rho_Y, rho_M) = {dyn_sum['F']:.4f}  "
      f"[{dyn_sum['F_ci'][0]:.4f}, {dyn_sum['F_ci'][1]:.4f}]")
idx2500d = np.argmin(np.abs(dyn_Nkept - 2500))
print(f"    sigma_F @ N=2500 = {dyn_Fstd[idx2500d]:.4f}")
print(f"")
print(f"  Delta_F (ps - dyn) = {delta_F:.4f}  (fidelity cost of feedforward)")
print(f"  Shot savings        = {trd['shot_cost_ratio']:.2f}x")
print(f"{'='*60}")
print(f"\n[✓] Done.")

  Fidelity–Shot-Cost Tradeoff Analysis
  Backend : ibm_torino
  Shots   : 10,000 per basis
  p_succ  : 0.2384  →  4.19× shot overhead

  Shots available — ps: 10,000  dyn: 10,000

  Running subsampling sweep (30 repeats per point) ...
   N_total    ps_Nkept      ps_F      ps_σ   dyn_Nkept     dyn_F     dyn_σ
  ------------------------------------------------------------------------
       200          45    0.8307    0.0595         200    0.7390    0.0495
       400          95    0.8409    0.0373         400    0.7513    0.0282
       600         144    0.8300    0.0267         600    0.7491    0.0177
       800         191    0.8412    0.0213         800    0.7421    0.0206
     1,000         243    0.8417    0.0197       1,000    0.7445    0.0172
     1,500         359    0.8398    0.0201       1,500    0.7471    0.0131
     2,000         477    0.8352    0.0139       2,000    0.7475    0.0091
     3,000         718    0.8353    0.0142       3,000    0.7457    0.0075
     4,000     

/var/folders/dg/bj9b922n29vcxp68247bp7vh0000gn/T/ipykernel_39793/2324118978.py:419: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
